# Notebook 2: Data Cleaning, Pre-Processing & Dimensionality Reduction
---
**Project:** Audio Recommendation Algorithm

**Input:** `/Users/saadult/Music Rec Algo/Music-Recommendation-Algorithm/data/train.csv` (raw, 28,362 songs)

**Output:** `data/train_cleaned.csv` (cleaned and scaled, ready for modeling)

**Goal:** Apply all decisions made during EDA to produce a clean, scaled dataset. Every column drop and transformation in this notebook is justified by a specific EDA finding.

### Steps in this notebook:
1. Drop identity and irrelevant columns
2. Drop low-variance features identified in EDA
3. Handle missing values
4. Remove outliers using IQR
5. Scale features using StandardScaler
6. Save cleaned dataset

---
## Setup: Imports and Data Loading

Import all required libraries and load the raw training data. We confirm the shape matches what we saw in EDA (28,362 rows × 24 columns) before making any changes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

%matplotlib inline
sns.set_style('whitegrid')

# Load raw training data
DATA_PATH = '/Users/saadult/Music Rec Algo/Music-Recommendation-Algorithm/data/train.csv'
df = pd.read_csv(DATA_PATH)

print(f'Raw data shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

---
## Step 1: Drop Identity and Irrelevant Columns

These columns are labels or metadata — they cannot be used by KMeans because they are either text or index values with no numeric meaning. The `lyrics` column is dropped per the assignment instructions; it requires a Word2Vec model (optional bonus challenge) to be useful. Keeping these columns would cause errors or meaningless distance calculations in the clustering step.

In [ ]:
# Drop identity and non-numeric columns
# 'Unnamed: 0'  — row index from original CSV, carries no information
# 'lyrics'      — raw text, requires NLP (Word2Vec bonus challenge)
# 'artist_name' — identity label, not a lyrical feature
# 'track_name'  — identity label, not a lyrical feature

cols_to_drop_identity = ['Unnamed: 0', 'lyrics', 'artist_name', 'track_name']
df_clean = df.drop(
    columns=[c for c in cols_to_drop_identity if c in df.columns]
)

print(f'Shape after dropping identity columns: {df_clean.shape}')
print(f'Remaining columns: {df_clean.columns.tolist()}')

---
## Step 2: Drop Low-Variance Features

**EDA justification:** The variance analysis in Notebook 1 identified four score columns with variance below 0.003. These columns are so heavily skewed toward zero that most songs look nearly identical on these dimensions. The KMeans algorithm measures distance between songs — if most songs score 0.001 on `shake the audience`, that column contributes almost nothing to separating any song from any other.

Columns being dropped and their variances:
- `shake the audience` — 0.00165
- `family/gospel` — 0.00176
- `family/spiritual` — 0.00260
- `dating` — 0.00274

Note: No columns are dropped for high correlation because EDA showed no pairs exceeded 0.27 — well below the 0.7 redundancy threshold.

In [ ]:
# Drop the four lowest-variance score columns identified in EDA
# These features lack sufficient spread to contribute meaningful cluster separation
low_variance_cols = [
    'shake the audience',  # variance: 0.00165
    'family/gospel',       # variance: 0.00176
    'family/spiritual',    # variance: 0.00260
    'dating',              # variance: 0.00274
]

df_clean = df_clean.drop(
    columns=[c for c in low_variance_cols if c in df_clean.columns]
)

print(f'Shape after dropping low-variance columns: {df_clean.shape}')
print(f'Remaining columns: {df_clean.columns.tolist()}')

---
## Step 3: Handle Missing Values

EDA confirmed zero null values across the entire dataset. We run the check again here on the cleaned dataframe as a verification step. If nulls are found at this stage, we drop those rows since the score columns come from a structured research paper and any null would indicate a data corruption issue rather than a systematic gap.

In [ ]:
# Verify no null values exist after column drops
nulls = df_clean.isnull().sum()
print('Missing values per column:')
print(
    nulls[nulls > 0]
    if nulls.sum() > 0
    else 'No missing values confirmed — no rows need to be dropped.'
)
print(f'\nTotal rows: {len(df_clean):,}')

In [ ]:
# Drop any null rows as a safety measure
# (Expected to remove 0 rows based on EDA findings)
rows_before = len(df_clean)
df_clean = df_clean.dropna()
print(f'Rows before: {rows_before:,}')
print(f'Rows after:  {len(df_clean):,}')
print(f'Rows removed: {rows_before - len(df_clean)}')

---
## Step 4: Remove Outliers Using IQR

Outliers are extreme values that sit far outside the normal range of a column. KMeans is sensitive to outliers because they can pull cluster centroids away from the true center of a group, distorting the results.

We use the **IQR (Interquartile Range) method**:
- IQR = the range of the middle 50% of values (Q3 − Q1)
- Any value more than 3× IQR beyond Q1 or Q3 is removed

We use 3× (instead of the standard 1.5×) because our columns are already bounded between 0 and 1 — extreme values are less extreme than in unbounded data.

In [ ]:
# Define the final set of score columns after all drops
# These are the features that will be fed into the KMeans model
score_cols_final = [
    'violence', 'world/life', 'night/time', 'romantic',
    'communication', 'obscene', 'music', 'movement/places',
    'light/visual perceptions', 'sadness', 'feelings'
]

# Keep only columns that exist in the dataframe
score_cols_final = [c for c in score_cols_final if c in df_clean.columns]
print(f'Score columns entering the model ({len(score_cols_final)} total):')
print(score_cols_final)

In [ ]:
# IQR outlier removal across all score columns
# Uses 3.0x multiplier (wider than standard 1.5x) due to 0-1 bounded data
rows_before = len(df_clean)

for col in score_cols_final:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 3.0 * iqr
    upper = q3 + 3.0 * iqr
    df_clean = df_clean[
        (df_clean[col] >= lower) & (df_clean[col] <= upper)
    ]

rows_after = len(df_clean)
print(f'Rows before outlier removal: {rows_before:,}')
print(f'Rows after outlier removal:  {rows_after:,}')
print(f'Rows removed: {rows_before - rows_after:,}')

---
## Step 5: Scale Features with StandardScaler

KMeans calculates the distance between every song and every cluster centroid. If one column has a much larger spread than another, it dominates those distance calculations — the other columns barely influence the result.

**EDA justification:** The variance analysis showed a 20× difference in spread between our strongest (`obscene`, variance 0.033) and weakest remaining features (`feelings`, variance 0.005). Without scaling, `obscene` would contribute 20× more than `feelings` to every distance calculation.

`StandardScaler` transforms each column to have **mean = 0** and **standard deviation = 1**, putting all features on equal footing. The shape of each distribution is preserved — only the scale changes.

**Important:** We fit the scaler on the training data here and save it. The same fitted scaler must be used on the recommendation data in Notebook 3.

In [ ]:
# Fit StandardScaler on the training score columns
# transform() re-centers each column to mean=0, std=1
scaler = StandardScaler()

X_scaled = pd.DataFrame(
    scaler.fit_transform(df_clean[score_cols_final]),
    columns=score_cols_final,
    index=df_clean.index
)

print('Before scaling — mean and std of first 3 columns:')
print(df_clean[score_cols_final[:3]].agg(['mean', 'std']).round(4))
print()
print('After scaling — should be ~0 mean and ~1 std:')
print(X_scaled[score_cols_final[:3]].agg(['mean', 'std']).round(4))

### Before vs After Scaling Visualization

This side-by-side histogram shows the effect of StandardScaler on the `sadness` column. The distribution shape is identical — scaling only shifts the x-axis so that 0 is now the mean and the spread is standardized to 1 standard deviation.

In [ ]:
# Visual comparison of a single column before and after scaling
# Note: the shape is identical — only the x-axis scale changes
col_to_show = 'sadness'

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(
    df_clean[col_to_show], bins=30, color='steelblue', edgecolor='white'
)
axes[0].set_title(f'{col_to_show} — BEFORE scaling', fontweight='bold')
axes[0].set_xlabel('Original value (0–1)')
axes[0].set_ylabel('Count')

axes[1].hist(
    X_scaled[col_to_show], bins=30, color='coral', edgecolor='white'
)
axes[1].set_title(f'{col_to_show} — AFTER scaling', fontweight='bold')
axes[1].set_xlabel('Scaled value (centered at 0)')
axes[1].set_ylabel('Count')

plt.suptitle(
    'Effect of StandardScaler — Shape preserved, scale changed',
    fontsize=13,
    fontweight='bold'
)
plt.tight_layout()
plt.show()

**Finding:** The distribution shape is identical before and after scaling. StandardScaler does not alter the underlying pattern — it only ensures all columns contribute equally to the KMeans distance calculations.

---
## Step 6: Build Final Dataframe

We combine the label columns (`genre`, `topic`, `release_date`, `age`, `len`) with the scaled score columns. The label columns are retained so we can interpret cluster results in human terms after modeling — they are not fed into the clustering algorithm itself.

In [ ]:
# Retain label columns for post-clustering interpretation
# These are NOT fed into KMeans — they are used to describe clusters afterward
label_cols = ['release_date', 'genre', 'topic', 'age', 'len']
label_cols = [c for c in label_cols if c in df_clean.columns]

# Combine label columns with scaled score columns
df_final = pd.concat(
    [
        df_clean[label_cols].reset_index(drop=True),
        X_scaled.reset_index(drop=True)
    ],
    axis=1
)

print(f'Final cleaned dataframe shape: {df_final.shape}')
print(f'Label columns retained: {label_cols}')
print(f'Score columns scaled: {score_cols_final}')
df_final.head()

---
## Step 7: Save the Cleaned Dataset

We save the cleaned, scaled dataframe as `train_cleaned.csv`. This file is the input for Notebook 3 (Modeling). The scaler object is also saved using `pickle` so the same scaling parameters can be applied to the recommendation dataset in Notebook 3 without re-fitting.

In [ ]:
import pickle
import os

# Save cleaned training data
OUTPUT_PATH = '/Users/saadult/Music Rec Algo/Music-Recommendation-Algorithm/data/train_cleaned.csv'
df_final.to_csv(OUTPUT_PATH, index=False)
print(f'Saved cleaned data to: {OUTPUT_PATH}')
print(f'Shape: {df_final.shape}')

# Save scaler so Notebook 3 can apply the same transformation to recommend.csv
SCALER_PATH = '/Users/saadult/Music Rec Algo/Music-Recommendation-Algorithm/data/scaler.pkl'
with open(SCALER_PATH, 'wb') as f:
    pickle.dump(scaler, f)
print(f'Saved scaler to: {SCALER_PATH}')

---
## Cleaning Summary

| Step | Action | Justification |
|---|---|---|
| Identity columns dropped | `Unnamed: 0`, `lyrics`, `artist_name`, `track_name` | Non-numeric, not usable by KMeans |
| Low-variance columns dropped | `shake the audience`, `family/gospel`, `family/spiritual`, `dating` | Variance < 0.003, confirmed in EDA |
| Null rows removed | 0 rows removed | EDA confirmed no missing values |
| Outliers removed | IQR method (3× multiplier) | KMeans is sensitive to extreme values |
| Features scaled | StandardScaler (mean=0, std=1) | 20× variance difference between features requires equalization |
| Final shape | ~28k rows × 16 columns | 11 scaled score cols + 5 label cols |

**Output file:** `data/train_cleaned.csv` — ready for KMeans modeling in Notebook 3.